In [1]:
import os
os.chdir("../")

In [2]:
import pandas as pd
import numpy as np

#  Create Synthetic Data (100 days of trading)
np.random.seed(42)
days = 100
spot_price = 2000 + np.cumsum(np.random.normal(0, 15, days))
# Futures usually track spot but with some "basis" drift
futures_price = spot_price + np.random.normal(5, 2, days) 

df = pd.DataFrame({
    "date": pd.date_range(start = "2024-01-01", periods = days),
    "gold_spot": spot_price,
    "gold_futures": futures_price
})

# Display first few rows
print("Preview of Data:")
print(df.head())

Preview of Data:
        date    gold_spot  gold_futures
0 2024-01-01  2007.450712   2009.619971
1 2024-01-02  2005.376748   2009.535457
2 2024-01-03  2015.092076   2019.406647
3 2024-01-04  2037.937524   2041.332969
4 2024-01-05  2034.425223   2039.102652


In [3]:
from QuantEngine.hedging import HedgeRatio

In [4]:
hedgeratio = HedgeRatio(
    pandas_df = df,
    spot_column_name = "gold_spot",
    future_column_name = "gold_futures"
)

In [5]:
h_star = hedgeratio.get_mhvr(cross_hedge = False)
effectiveness = hedgeratio.get_hedge_effectiveness(cross_hedge = False)

In [6]:
print(f"Optimal Hedge Ratio (h*): {h_star}")
print(f"Hedge Effectiveness: {effectiveness['hedge_effectiveness'] * 100}%")
print(f"Effectiveness Rating: {effectiveness['effectiveness_rating']}")

Optimal Hedge Ratio (h*): 0.9609
Hedge Effectiveness: 95.95%
Effectiveness Rating: STRONG


In [7]:
sizing_result = hedgeratio.calculate_sizing(
    portfolio_value = 1_000_000,
    current_futures_price = 2100,
    contract_size = 100,
    cross_hedge = False
)

In [12]:
print(f"Action: {sizing_result.get('direction')} {sizing_result.get('contracts_to_trade')} Contracts")
print(f"Hedge Notional: ${sizing_result.get('hedge_notional')}")

Action: SHORT 5 Contracts
Hedge Notional: $1050000
